# APD Kaggle generation runner

This notebook generates one deterministic APD shard on a free GPU runtime. It does not change the scientific grid; it reads `results/missing_generation_manifest_2026-06-02.csv` and writes merge-compatible metadata shards plus images into a downloadable ZIP.

Use only free runtimes. Do not enable paid tiers, background servers, or remote-control workarounds.

In [ ]:
# === Configuration ===
REPO_URL = "https://github.com/hlaverde/apd-audit.git"
REPO_DIR = "/kaggle/working/apd-audit"
OUTPUT_ROOT = "/kaggle/working/apd_cloud_output"

MODEL = "runwayml/stable-diffusion-v1-5"
LANGUAGE = "en"
GRID = "main"
SHARD_ID = 0
N_SHARDS = 4
MAX_IMAGES_PER_RUN = 50
CHECKPOINT_EVERY = 5
CLASSIFY = True
DRY_RUN = False

RUNNER = "kaggle"
RUN_ID = f"{RUNNER}_{MODEL.split('/')[-1].replace('-', '_')}_{LANGUAGE}_s{SHARD_ID}of{N_SHARDS}"


In [ ]:
# Kaggle writes artifacts under /kaggle/working so they are downloadable from the Output panel.


In [ ]:
# Clone or update repository.
import os, pathlib, subprocess, sys
repo = pathlib.Path(REPO_DIR)
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
os.chdir(repo)
print('Repo:', repo)


In [ ]:
# Install runtime dependencies. Heavy ML packages are required for local diffusers on GPU.
import subprocess, sys
base_packages = [
    'pandas>=2.2,<3', 'pyarrow>=15', 'numpy>=1.26,<3',
    'pydantic>=2.7', 'pydantic-settings>=2.3', 'python-dotenv>=1',
    'pillow>=10', 'requests>=2.32', 'httpx>=0.27',
]
ml_packages = [
    'torch>=2.2', 'diffusers>=0.30', 'transformers>=4.42',
    'accelerate>=0.33', 'safetensors>=0.4',
]
cv_packages = [
    'opencv-python-headless>=4.9', 'mediapipe>=0.10.14', 'skin-tone-classifier>=1.2.3',
]
packages = base_packages + ml_packages + (cv_packages if CLASSIFY else [])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)


In [ ]:
# Show manifest availability and selected pending count before running.
import hashlib, pandas as pd, pathlib
manifest_path = pathlib.Path('results/missing_generation_manifest_2026-06-02.csv')
if not manifest_path.exists():
    raise FileNotFoundError(f'Manifest missing: {manifest_path}. Generate it locally and commit/copy it first.')
manifest = pd.read_csv(manifest_path)
def shard(image_id, n):
    return int(hashlib.sha256(str(image_id).encode('utf-8')).hexdigest()[:16], 16) % n
preview = manifest[(manifest.model == MODEL) & (manifest.language == LANGUAGE) & (manifest.grid == GRID) & (manifest.prompt_status == 'ok')].copy()
preview = preview[preview.image_id.map(lambda x: shard(x, N_SHARDS) == SHARD_ID)]
print('Pending rows in selected slice:', len(preview))
display(preview.head(10))


In [ ]:
# Run generation. Checkpoints are written every CHECKPOINT_EVERY rows.
import subprocess, sys
cmd = [
    sys.executable, 'scripts/cloud_generation_runner.py',
    '--manifest', 'results/missing_generation_manifest_2026-06-02.csv',
    '--existing-metadata', 'images/main/metadata.parquet',
    '--output-root', OUTPUT_ROOT,
    '--runner', RUNNER,
    '--run-id', RUN_ID,
    '--model', MODEL,
    '--language', LANGUAGE,
    '--grid', GRID,
    '--shard-id', str(SHARD_ID),
    '--n-shards', str(N_SHARDS),
    '--max-images-per-run', str(MAX_IMAGES_PER_RUN),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
]
cmd.append('--classify' if CLASSIFY else '--no-classify')
if DRY_RUN:
    cmd.append('--dry-run')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# List downloadable outputs.
import pathlib
out = pathlib.Path(OUTPUT_ROOT)
for p in sorted(out.rglob('*')):
    if p.is_file():
        print(p, p.stat().st_size)


## Resume behavior

If the runtime disconnects, reconnect with the same configuration and run all cells again. The runner reads existing local shard metadata and skips completed `image_id`s, then continues the same deterministic slice.